# 📘 EBIAS: ECC-Enabled Blockchain-Based Identity Authentication Scheme
## Section 4.3 — Security Analysis: All 10 Propositions
**Paper:** *EBIAS: ECC-enabled blockchain-based identity authentication scheme for IoT device*  
**Authors:** Wenyue Wang, Biwei Yan, Baobao Chai, Ruiyao Shen, Anming Dong, Jiguo Yu  
**Journal:** High-Confidence Computing 5 (2025) 100240

---

## 🔍 What is Section 4.3 About?

Section 4.3 is the **Security Analysis** of the EBIAS scheme. After the authors designed the authentication protocol in Sections 4.1 and 4.2, they now need to *prove* that it is secure.

### Paragraph 1 — Overview of the Security Goal

> *"The proposed solution utilizes blockchain technology for device identity authentication, combined with the public–private key system of the ECC algorithm and the hash function of the SHA-256 algorithm, to ensure the uniqueness, integrity, and security of device identity."*

**Plain English:** The system uses 3 combined tools to make authentication secure:
1. **Blockchain** → Decentralized storage of identity. No single point of failure.
2. **ECC (Elliptic Curve Cryptography)** → Public/private key pairs for each device. Very secure with small key sizes.
3. **SHA-256** → A one-way hash function. Once hashed, you can't reverse it.

**Example:**  
Imagine your IoT smart lock. Its identity is:
- Stored across 100 blockchain nodes (not one central server) ✅
- Has a private key only it knows, and a public key everyone can see ✅
- Its password is hashed — even if intercepted, attacker sees garbage ✅

### Paragraph 2 — How They Prove Security

> *"We will assess the security of the proposed solution by examining various intrusion techniques. We illustrate EBIAS ability to offer the subsequent security capabilities and safeguard against the following types of attacks."*

**Plain English:** The authors prove security by listing 10 attack scenarios and showing EBIAS blocks each one. Each proof is called a **Proposition**. Think of it like a lawyer saying: "Can the attacker do X? No, because of Y."

---

# 🔐 Setup: ECC Cryptographic Building Blocks

Before diving into propositions, we implement the core ECC math used throughout. This is the mathematical foundation.

## 📐 Key Equation Reference Table

| Symbol | Full Name | Meaning |
|--------|-----------|----------|
| `G` | Generator Point | A fixed, publicly known point on the elliptic curve. Acts as the "base" for all key operations. |
| `n` | Order of G | The number of times you can add G to itself before cycling back. G is chosen so n is a large prime. |
| `priK` | Private Key | A secret random integer chosen by device. Only device knows this. |
| `pubK` | Public Key | `priK * G`. A point on the curve. Safe to share publicly. |
| `N1`, `N2` | Random Nonces | One-time random numbers chosen fresh for each session. |
| `CK` | Encryption Key | Derived from registration. Used to protect comms between device and IAC. |
| `CK'` | ECC form of CK | `CK * G` — the ECC point version of CK. |
| `P1`, `P2`, `P3`, `P4` | Session Points | ECC points computed during auth. Used to prove identity without revealing secrets. |
| `Y`, `Z`, `Vi` | Hash Verifiers | SHA-256 hashes of combined values. Sent to prove authenticity. |
| `SK` | Session Key | Final shared key established after mutual auth. Encrypts the session. |
| `IDi` | Device ID | The unique identifier of an IoT device. |
| `PWi` | Device Password | The access credential/password of the device. |
| `Ii` | Identity Hash | `h(IDi ∥ PWi)` — SHA-256 hash of ID+password combined. |
| `PIDi` | Pseudonym ID | A fake identity for device, changes each session. Protects real ID. |
| `Ri` | Random Sequence | Random number generated by IAC during registration. |
| `X` | IAC Secret Key | The master secret key held only by the Identity Authentication Center. |
| `Ti` | Stored Token | `Ri ⊕ h(X ∥ PIDi)` — stored on blockchain, used to recover Ri later. |
| `Ai` | Auth Token | `h(Ti ⊕ Ii ⊕ CK')` — a hash token linking device's secret to the session. |
| `A'i` | ECC Auth Point | `Ai * G` — ECC point version of Ai. |
| `T1..T5` | Timestamps | Time values added to messages to prevent replay attacks. |
| `∆T` | Time Threshold | Maximum allowed delay between sending/receiving a message. |
| `h(.)` | SHA-256 Function | One-way hash. Input → fixed 256-bit output. Cannot be reversed. |
| `∥` | Concatenation | Join two values together. `A∥B` means "A followed by B". |
| `⊕` | XOR Operation | Bitwise exclusive OR. `A⊕A = 0`. Used to hide/reveal values. |

In [1]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
# Run this cell first!
import subprocess
subprocess.run(['pip', 'install', 'pycryptodome', 'tinyec'], capture_output=True)
print("✅ Dependencies ready")

✅ Dependencies ready


In [2]:
# ============================================================
# CELL 2: Core ECC & Hash Utilities
# We simulate ECC using Python integers (mod prime arithmetic)
# This mirrors the actual ECC operations in the paper
# ============================================================

import hashlib
import secrets
import time

# ---- Simplified ECC over a small prime for demonstration ----
# In real EBIAS: curve is secp256k1 or similar with 160-bit keys
# Here we use toy parameters to show the logic clearly

class ECC_Point:
    """Represents a point on an elliptic curve y^2 = x^3 + ax + b (mod p)"""
    
    # Curve parameters (toy example for demonstration)
    # In EBIAS paper: 160-bit keys, 320-bit ECC points
    p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141  # secp256k1 order
    
    def __init__(self, value):
        """We represent points as integers for simplicity"""
        self.value = value % self.p
    
    def __mul__(self, scalar):
        """Scalar multiplication: point * k"""
        # In real ECC this is repeated point addition
        # We simulate with modular multiplication
        return ECC_Point((self.value * scalar) % self.p)
    
    def __rmul__(self, scalar):
        return self.__mul__(scalar)
    
    def __repr__(self):
        return f"ECCPoint({hex(self.value)[:18]}...)"
    
    def to_bytes(self):
        return self.value.to_bytes(32, 'big')


# G = Generator Point (publicly known base point on elliptic curve)
# In secp256k1 this is a specific (x,y) coordinate. We use its order as value.
G = ECC_Point(0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798)

print(f"🔷 Generator Point G = {G}")
print(f"   G represents the base point of the elliptic curve")
print(f"   All keys are multiples of G: pubK = priK * G")


def sha256_hash(*args):
    """
    h(.) — SHA-256 hash function
    Takes any number of arguments, concatenates them, returns hash as int
    
    Paper notation: h(A∥B) = sha256_hash(A, B)
    """
    combined = b''
    for arg in args:
        if isinstance(arg, int):
            combined += arg.to_bytes(32, 'big')
        elif isinstance(arg, ECC_Point):
            combined += arg.to_bytes()
        elif isinstance(arg, str):
            combined += arg.encode()
        elif isinstance(arg, bytes):
            combined += arg
    return int(hashlib.sha256(combined).hexdigest(), 16)


def xor_values(a, b):
    """
    ⊕ — XOR operation
    Paper uses this to hide values: A ⊕ B
    To recover A: (A ⊕ B) ⊕ B = A
    """
    if isinstance(a, ECC_Point): a = a.value
    if isinstance(b, ECC_Point): b = b.value
    return a ^ b


def get_timestamp():
    """T1, T2, T3, T4, T5 — Unix timestamps"""
    return int(time.time() * 1000)  # milliseconds


DELTA_T = 5000  # ∆T = 5 seconds (5000ms) allowed time window

print("\n✅ ECC utilities loaded successfully")
print(f"   SHA-256 produces 256-bit (32-byte) hashes")
print(f"   ∆T (time threshold) = {DELTA_T}ms")

🔷 Generator Point G = ECCPoint(0x79be667ef9dcbbac...)
   G represents the base point of the elliptic curve
   All keys are multiples of G: pubK = priK * G

✅ ECC utilities loaded successfully
   SHA-256 produces 256-bit (32-byte) hashes
   ∆T (time threshold) = 5000ms


---
# 📋 Stage 3 & 4 Recap: Registration + Authentication Math

Before proving propositions, let's implement the full registration and authentication so we can reference live values.

## Stage 3: Device Registration

### Step 1 — Device computes Ii and sends to IAC
$$I_i = h(ID_i \| PW_i)$$

- **IDi** = Device's unique identifier (e.g., "SmartLock_001")
- **PWi** = Device's password/credential (e.g., "secretpass123")
- **Ii** = SHA-256 hash of their concatenation
- **Why hash?** So the actual ID and password are never sent in plaintext

### Step 2 — IAC generates registration tokens

$$PID_i = h(R_i \| ID_{IAC} \| I_i) \oplus ID_{IAC}$$
- **Ri** = Random sequence (fresh random number from IAC)
- **IDIAC** = IAC's own identity
- **PIDi** = Pseudonym identity — protects real ID

$$CK = h(R_i \| X \| EXP\_time \| PID_i)$$
- **X** = IAC's master secret key
- **EXP_time** = Certificate expiry time
- **CK** = Encryption key for future sessions

$$CK' = CK * G$$
- **CK'** = ECC point version of CK (sent to device, safe to store)

$$T_i = R_i \oplus h(X \| PID_i)$$
- **Ti** = XOR-masked version of Ri, stored on blockchain
- IAC can later recover Ri = Ti ⊕ h(X ∥ PIDi)

$$A_i = h(T_i \oplus I_i \oplus CK')$$
- **Ai** = Authentication token linking device secret to session

$$A'_i = A_i * G$$
- **A'i** = ECC point of Ai, stored by IAC

In [3]:
# ============================================================
# CELL 3: Stage 3 — Device Registration Implementation
# ============================================================

print("=" * 60)
print("STAGE 3: DEVICE REGISTRATION")
print("=" * 60)

# --- Device Information ---
IDi  = "SmartLock_Device_001"     # Device unique identifier
PWi  = "SecurePass#2024"          # Device password/credential
IDIAC = "IAC_Server_MAIN"         # IAC's identity
X    = secrets.randbits(160)      # IAC master secret key (NEVER shared)
EXP_time = get_timestamp() + (365 * 24 * 3600 * 1000)  # 1 year expiry

print(f"\n📱 Device ID  (IDi)  : {IDi}")
print(f"🔑 Password   (PWi)  : {PWi}")
print(f"🏢 IAC ID (IDIAC)    : {IDIAC}")
print(f"🔐 IAC Secret (X)    : {hex(X)[:20]}... [NEVER shared]")

# --- STEP 1: Device computes Ii = h(IDi ∥ PWi) ---
# Ii is the identity hash — combines device ID and password
Ii = sha256_hash(IDi, PWi)
print(f"\n--- STEP 1: Device computes Ii ---")
print(f"Ii = h(IDi ∥ PWi)")
print(f"Ii = h('{IDi}' ∥ '{PWi}')")
print(f"Ii = {hex(Ii)[:30]}...")
print(f"📌 Ii is sent to IAC via secure channel (not the raw IDi/PWi)")

# --- STEP 2: IAC generates PIDi, CK, CK', Ti, Ai, A'i ---
Ri = secrets.randbits(160)  # IAC generates random sequence

# PIDi = h(Ri ∥ IDIAC ∥ Ii) ⊕ IDIAC
IDIAC_int = int(hashlib.sha256(IDIAC.encode()).hexdigest(), 16)
PIDi = xor_values(sha256_hash(Ri, IDIAC, Ii), IDIAC_int)

# CK = h(Ri ∥ X ∥ EXP_time ∥ PIDi)
CK = sha256_hash(Ri, X, EXP_time, PIDi)

# CK' = CK * G  (ECC point — safe to send to device)
CK_prime = CK * G

# Ti = Ri ⊕ h(X ∥ PIDi)  — stored on blockchain
Ti = xor_values(Ri, sha256_hash(X, PIDi))

# Ai = h(Ti ⊕ Ii ⊕ CK')
Ai_val = xor_values(xor_values(Ti, Ii), CK_prime.value)
Ai = sha256_hash(Ai_val)

# A'i = Ai * G
Ai_prime = Ai * G

print(f"\n--- STEP 2: IAC generates registration tokens ---")
print(f"Ri     = {hex(Ri)[:20]}... (random, stays with IAC)")
print(f"PIDi   = {hex(PIDi)[:20]}... (pseudonym, sent to device)")
print(f"CK     = {hex(CK)[:20]}... (encryption key, IAC keeps)")
print(f"CK'    = {CK_prime}  (ECC point of CK, sent to device)")
print(f"Ti     = {hex(Ti)[:20]}... (stored on blockchain)")
print(f"Ai     = {hex(Ai)[:20]}... (auth token, IAC computes)")
print(f"A'i    = {Ai_prime}  (ECC point of Ai, stored by IAC)")

# Device stores: {PIDi, CK', Ti}
# IAC stores:    {A'i, PIDi, EXP_time}
# Blockchain stores: {PIDi, Ti}

print(f"\n📦 Device stores    : PIDi, CK', Ti")
print(f"📦 IAC stores       : A'i, PIDi, EXP_time")
print(f"🔗 Blockchain stores: PIDi, Ti")
print("\n✅ Registration Complete!")

STAGE 3: DEVICE REGISTRATION

📱 Device ID  (IDi)  : SmartLock_Device_001
🔑 Password   (PWi)  : SecurePass#2024
🏢 IAC ID (IDIAC)    : IAC_Server_MAIN
🔐 IAC Secret (X)    : 0xe6936836e8c74dfffb... [NEVER shared]

--- STEP 1: Device computes Ii ---
Ii = h(IDi ∥ PWi)
Ii = h('SmartLock_Device_001' ∥ 'SecurePass#2024')
Ii = 0x3f77524b66811e3e43b49a946f41...
📌 Ii is sent to IAC via secure channel (not the raw IDi/PWi)

--- STEP 2: IAC generates registration tokens ---
Ri     = 0xdae8474895c0f7b09f... (random, stays with IAC)
PIDi   = 0xb3f81f4a46fd957ede... (pseudonym, sent to device)
CK     = 0x2a77a07a16e29470ec... (encryption key, IAC keeps)
CK'    = ECCPoint(0x46c3196cae6302af...)  (ECC point of CK, sent to device)
Ti     = 0x1462589d2401cdffd2... (stored on blockchain)
Ai     = 0x83e959b1a9795b8743... (auth token, IAC computes)
A'i    = ECCPoint(0xd7e96eae4355da4c...)  (ECC point of Ai, stored by IAC)

📦 Device stores    : PIDi, CK', Ti
📦 IAC stores       : A'i, PIDi, EXP_time
🔗 Blockcha

---
## Stage 4: Identity Authentication — The 6 Steps

### Step 1 — Device → IAC Login Request

$$P_1 = N_1 * G$$
- **N1** = Fresh random number chosen by device for THIS session only
- **G** = Generator point (public, everyone knows it)
- **P1** = ECC point = N1 multiplied by G
- **Why?** N1 is secret. P1 is public. From P1, attacker CANNOT find N1 (this is the ECDLP hardness)

$$P_2 = h(N_1 * CK')$$
- **N1 * CK'** = scalar × ECC point = another ECC point (Diffie-Hellman style)
- **P2** = hash of that shared point
- **Why?** This proves device has CK' without revealing it

$$Y = h(P_1 \| P_2 \| T_1)$$
- **T1** = Timestamp (freshness proof)
- **Y** = Hash of P1, P2, and timestamp
- **Why?** This is the authentication token. Only someone with N1 and CK' can compute Y correctly.

Device sends: **{PIDi, P1, Y, T1}** → IAC

In [4]:
# ============================================================
# CELL 4: Stage 4 — Full Authentication (6 Steps)
# ============================================================

print("=" * 60)
print("STAGE 4: IDENTITY AUTHENTICATION")
print("=" * 60)

# ---- STEP 1: Device generates login request ----
print("\n--- STEP 1: Device → IAC (Login Request) ---")

N1 = secrets.randbits(160)    # Random nonce for this session
T1 = get_timestamp()          # Timestamp T1

# P1 = N1 * G
# N1 is the private random. P1 is the ECC point.
# Attacker sees P1 but CANNOT reverse it to find N1 (ECDLP problem)
P1 = N1 * G

# P2 = h(N1 * CK')
# This is like Diffie-Hellman: shared secret = private * (other's public point)
N1_CK_prime = N1 * CK_prime   # = N1 * CK * G  (only device with N1 and CK' can compute)
P2 = sha256_hash(N1_CK_prime)

# Y = h(P1 ∥ P2 ∥ T1)
Y = sha256_hash(P1, P2, T1)

print(f"N1     = {hex(N1)[:20]}... (secret random, device only)")
print(f"T1     = {T1} (timestamp)")
print(f"P1     = N1 * G = {P1}")
print(f"P2     = h(N1 * CK') = {hex(P2)[:20]}...")
print(f"Y      = h(P1 ∥ P2 ∥ T1) = {hex(Y)[:20]}...")
print(f"\n📤 Device sends to IAC: {{PIDi, P1, Y, T1}}")

# ---- STEP 2: IAC verifies device request ----
print("\n--- STEP 2: IAC Verifies Login Request ---")
import time as time_module
time_module.sleep(0.01)  # Simulate network delay
T2 = get_timestamp()

# Check timestamp freshness
if T2 - T1 < DELTA_T:
    print(f"✅ Timestamp check: T2-T1 = {T2-T1}ms < ∆T={DELTA_T}ms → FRESH")
else:
    print(f"❌ REPLAY ATTACK DETECTED: T2-T1 = {T2-T1}ms > ∆T={DELTA_T}ms")

# IAC recovers Rs = Ti ⊕ h(X ∥ PIDi)
Rs = xor_values(Ti, sha256_hash(X, PIDi))
print(f"Rs     = Ti ⊕ h(X ∥ PIDi) = {hex(Rs)[:20]}... (should equal Ri)")
print(f"Ri     = {hex(Ri)[:20]}...")
print(f"Rs==Ri : {Rs == Ri} ← IAC successfully recovered the original random!")

# IAC recomputes CK
CK_iac = sha256_hash(Rs, X, EXP_time, PIDi)

# IAC computes P2' = h(P1 * CK)
P1_CK = P1 * CK_iac   # = N1 * G * CK = N1 * CK * G  (same as device's N1 * CK')
P2_prime = sha256_hash(P1_CK)

# IAC verifies Y' = h(P1 ∥ P2' ∥ T1) == Y
Y_prime = sha256_hash(P1, P2_prime, T1)
print(f"\nP2'    = h(P1 * CK) = {hex(P2_prime)[:20]}...")
print(f"Y'     = h(P1 ∥ P2' ∥ T1) = {hex(Y_prime)[:20]}...")
print(f"Y==Y'  : {Y == Y_prime} ← {'✅ Device AUTHENTICATED by IAC' if Y==Y_prime else '❌ AUTH FAILED'}")

# ---- STEP 3: IAC → Device (Challenge) ----
print("\n--- STEP 3: IAC → Device (Authentication Challenge) ---")
N2 = secrets.randbits(160)    # IAC's fresh random nonce
T3 = get_timestamp()

# P3 = N2 * G
P3 = N2 * G

# P4 = N2 * A'i  (IAC uses its stored A'i)
P4 = N2 * Ai_prime

# Z = h(P3 ∥ P4 ∥ T3)
Z = sha256_hash(P3, P4, T3)

print(f"N2     = {hex(N2)[:20]}... (IAC's secret random)")
print(f"P3     = N2 * G = {P3}")
print(f"P4     = N2 * A'i = {P4}")
print(f"Z      = h(P3 ∥ P4 ∥ T3) = {hex(Z)[:20]}...")
print(f"\n📤 IAC sends to Device: {{Z, P3, T3}}")

# ---- STEP 4: Device verifies IAC ----
print("\n--- STEP 4: Device Verifies IAC Identity ---")
time_module.sleep(0.01)
T4 = get_timestamp()

if T4 - T3 < DELTA_T:
    print(f"✅ Timestamp check: T4-T3 = {T4-T3}ms < ∆T={DELTA_T}ms → FRESH")

# Device recomputes Ai = h(Ti ⊕ Ii ⊕ CK')
Ai_device_val = xor_values(xor_values(Ti, Ii), CK_prime.value)
Ai_device = sha256_hash(Ai_device_val)

# P4' = P3 * Ai  (device uses its computed Ai)
P4_prime = P3 * Ai_device

# Z' = h(P3 ∥ P4' ∥ T3)
Z_prime = sha256_hash(P3, P4_prime, T3)

print(f"Ai     (device) = {hex(Ai_device)[:20]}...")
print(f"P4'    = P3 * Ai = {P4_prime}")
print(f"Z'     = h(P3 ∥ P4' ∥ T3) = {hex(Z_prime)[:20]}...")
print(f"Z==Z'  : {Z == Z_prime} ← {'✅ IAC AUTHENTICATED by Device' if Z==Z_prime else '❌ IAC AUTH FAILED'}")

# ---- STEP 5: Device → IAC (Final Confirmation) ----
print("\n--- STEP 5: Device → IAC (Final Confirmation) ---")

# SK = h((N1 * P3) ∥ Ai ∥ T4)
N1_P3 = N1 * P3   # = N1 * N2 * G  (Diffie-Hellman shared secret)
SK = sha256_hash(N1_P3, Ai_device, T4)

# Vi = h(SK ∥ (N1 * CK'))
Vi = sha256_hash(SK, N1_CK_prime)

print(f"N1*P3  = N1 * N2 * G = {N1_P3}  ← Diffie-Hellman shared secret!")
print(f"SK     = h(N1*P3 ∥ Ai ∥ T4) = {hex(SK)[:20]}...  ← SESSION KEY")
print(f"Vi     = h(SK ∥ N1*CK') = {hex(Vi)[:20]}...")
print(f"\n📤 Device sends to IAC: {{Vi, T4}}")

# ---- STEP 6: IAC verifies and establishes session ----
print("\n--- STEP 6: IAC Final Verification & Session Key ---")
time_module.sleep(0.01)
T5 = get_timestamp()

if T5 - T4 < DELTA_T:
    print(f"✅ Timestamp check: T5-T4 = {T5-T4}ms < ∆T={DELTA_T}ms → FRESH")

# IAC computes Vi' = h((P1 * CK) ∥ SK)
# Note: P1 * CK = N1 * G * CK = N1 * CK * G = N1 * CK'  (same shared secret)
P1_CK_point = P1 * CK_iac
Vi_prime = sha256_hash(P1_CK_point, SK)

# IAC's SK' = h((N2 * P1) ∥ Ai ∥ T4)
N2_P1 = N2 * P1   # = N2 * N1 * G = N1 * N2 * G  (same as device's N1 * P3)
SK_prime = sha256_hash(N2_P1, Ai, T4)

print(f"Vi'    = h(P1*CK ∥ SK) = {hex(Vi_prime)[:20]}...")
print(f"Vi==Vi': {Vi == Vi_prime} ← {'✅ MUTUAL AUTH SUCCESS' if Vi==Vi_prime else '❌ FAILED'}")
print(f"\nSK  (device) = {hex(SK)[:20]}...")
print(f"SK' (IAC)    = {hex(SK_prime)[:20]}...")
print(f"SK==SK': {SK == SK_prime} ← {'✅ SESSION KEY ESTABLISHED' if SK==SK_prime else '❌ KEY MISMATCH'}")
print(f"\n🎉 AUTHENTICATION COMPLETE! Session encrypted with SK.")

STAGE 4: IDENTITY AUTHENTICATION

--- STEP 1: Device → IAC (Login Request) ---
N1     = 0x1a77a15a0856214120... (secret random, device only)
T1     = 1772603117946 (timestamp)
P1     = N1 * G = ECCPoint(0xda38ea8abc65af16...)
P2     = h(N1 * CK') = 0x9b9a9593d116151a75...
Y      = h(P1 ∥ P2 ∥ T1) = 0xfc996bdb49252ab796...

📤 Device sends to IAC: {PIDi, P1, Y, T1}

--- STEP 2: IAC Verifies Login Request ---
✅ Timestamp check: T2-T1 = 24ms < ∆T=5000ms → FRESH
Rs     = Ti ⊕ h(X ∥ PIDi) = 0xdae8474895c0f7b09f... (should equal Ri)
Ri     = 0xdae8474895c0f7b09f...
Rs==Ri : True ← IAC successfully recovered the original random!

P2'    = h(P1 * CK) = 0x9b9a9593d116151a75...
Y'     = h(P1 ∥ P2' ∥ T1) = 0xfc996bdb49252ab796...
Y==Y'  : True ← ✅ Device AUTHENTICATED by IAC

--- STEP 3: IAC → Device (Authentication Challenge) ---
N2     = 0x423714403b0c46c53d... (IAC's secret random)
P3     = N2 * G = ECCPoint(0x758c640bf12cb080...)
P4     = N2 * A'i = ECCPoint(0x33de93e5ff932f10...)
Z      = h(P

---
# 🛡️ PROPOSITION 1: EBIAS Resists Password/Identity Guessing

## What the Author Says (Paragraph by Paragraph)

**Paragraph 1:**  
> *"In EBIAS, during the device registration phase, Di transmits Ii (where Ii = h(IDi∥PWi)) to IAC through a secure channel. Therefore, due to the irreversible nature of the hash function, an attacker A cannot compute Di and PWi."*

**Meaning:** The device never sends its raw password. It sends only `Ii = h(IDi ∥ PWi)`. Since SHA-256 is a **one-way function**, attacker cannot reverse Ii to get IDi or PWi.

**Example:** If IDi = "Device_001" and PWi = "pass123", then Ii = SHA256("Device_001pass123") = some 256-bit number. You cannot go backwards from Ii to "pass123".

**Paragraph 2:**  
> *"Let's assume that A intercepts the information {PIDi, P1, Y, T1} when Di logs in, where P1 = N1 ∗ G, P2 = h(N1 ∗ CK′), Y = h(P1∥P2∥T1)."*

**Meaning:** Even if attacker captures the full login message, they still can't guess the password because:
- P1 = N1 * G → Cannot find N1 from P1 (ECDLP hardness)
- P2 = h(N1 * CK') → Would need both N1 and CK' to compute
- PIDi is a pseudonym protected by hash+XOR

## Key Terms in Proposition 1

| Term | Formula | Explanation |
|------|---------|-------------|
| **Ii** | `h(IDi ∥ PWi)` | Identity hash. Hash of device ID + password. Never reversible. |
| **P1** | `N1 * G` | N1 is secret random. G is public. P1 is public. N1 cannot be found from P1. |
| **P2** | `h(N1 * CK')` | Requires knowing N1 AND CK'. Attacker has neither. |
| **Y** | `h(P1 ∥ P2 ∥ T1)` | Verification hash. Requires P1, P2, and fresh timestamp. |
| **PIDi** | `h(Ri ∥ IDIAC ∥ Ii) ⊕ IDIAC` | Pseudonym. Protected by random Ri, hash, and XOR. |

## How to Read P1 = N1 * G

- **P1** = the result (a point on the elliptic curve)
- **N1** = a large random integer chosen by the device (kept secret)
- **`*`** = ECC scalar multiplication (NOT regular multiplication) — means "add G to itself N1 times"
- **G** = the generator point (a fixed, publicly known point on the curve)

**Analogy:** Think of a clock face. G is 12 o'clock. N1 = 5 means "go 5 steps clockwise → land on 5 o'clock". You can easily go forward (compute P1 from N1), but you cannot figure out how many steps were taken just by looking at where the hand is pointing (that's the ECDLP).

In [ ]:
# ============================================================
# CELL 5: Proposition 1 — Resistance to Password Guessing
# ============================================================

print("=" * 60)
print("PROPOSITION 1: RESISTANCE TO PASSWORD GUESSING")
print("=" * 60)

# Simulate an attacker who intercepts {PIDi, P1, Y, T1}
# Attacker tries to guess the password

intercepted = {
    'PIDi': PIDi,
    'P1': P1,
    'Y': Y,
    'T1': T1
}
print(f"\n🔍 Attacker intercepts: {{PIDi, P1, Y, T1}}")
print(f"  PIDi = {hex(PIDi)[:20]}... (pseudonym, not real ID)")
print(f"  P1   = {P1} (ECC point, public)")
print(f"  Y    = {hex(Y)[:20]}... (hash verifier)")
print(f"  T1   = {T1}")

print(f"\n--- Attacker tries to guess IDi and PWi ---")

# Attacker tries common passwords
guesses = ["password", "123456", "admin", "iot_device", "SecurePass#2024"]
target_IDi = "SmartLock_Device_001"  # Even if attacker knows IDi somehow

print(f"Password guesses:")
found = False
for guess in guesses:
    Ii_guess = sha256_hash(target_IDi, guess)
    # Attacker would need to compute P2 and then Y with this Ii_guess
    # But they don't have N1 or CK', so they CANNOT compute P2
    # They cannot verify if their guess is correct!
    print(f"  Trying '{guess}': Ii_guess = {hex(Ii_guess)[:20]}...")
    # Even if guess is correct, attacker can't confirm without CK'
    if guess == PWi:
        print(f"    ↑ This IS the correct password — but attacker CAN'T KNOW that!")
        print(f"    Because without CK', they cannot compute P2 to verify against Y.")
        found = True

print(f"\n🔒 RESULT: Even with correct Ii, attacker cannot compute:")
print(f"   P2 = h(N1 * CK')  ← requires N1 (unknown) AND CK' (never transmitted in plaintext)")
print(f"   Y  = h(P1∥P2∥T1)  ← requires valid P2")
print(f"\n✅ PROPOSITION 1 PROVEN: Password guessing attack FAILS.")
print(f"   Reason: SHA-256 is one-way + attacker lacks N1 and CK'")

---
# 🛡️ PROPOSITION 2: EBIAS Resists Replay Attacks

## What is a Replay Attack?
An attacker records a valid authentication message `{PIDi, P1, Y, T1}` and **replays it later** to fool the IAC into thinking the device is authentic again.

## How EBIAS Blocks It

**Paragraph 1:** Even if replayed, the attacker cannot compute Ai (because they don't know Ii), so they can't complete the mutual auth challenge.

**Paragraph 2:** Every message includes a timestamp T1. IAC checks:
$$T_2 - T_1 < \Delta T$$
- **T2** = IAC's current time when message is received
- **T1** = Timestamp embedded in the message
- **∆T** = Maximum allowed delay (e.g., 5 seconds)

If the message is replayed even 10 seconds later, `T2 - T1 > ∆T` → IAC rejects it!

**Example:** You record a valid login at 10:00:00. You replay it at 10:00:10. IAC says: "This message claims to be from 10:00:00, but it's now 10:00:10 — that's 10 seconds old, rejected!"

In [ ]:
# ============================================================
# CELL 6: Proposition 2 — Replay Attack Resistance
# ============================================================

print("=" * 60)
print("PROPOSITION 2: RESISTANCE TO REPLAY ATTACKS")
print("=" * 60)

import time as time_module

# Attacker records a valid message
recorded_message = {'PIDi': PIDi, 'P1': P1, 'Y': Y, 'T1': T1}
print(f"\n🔴 Attacker records valid message at T1 = {T1}")

def iac_timestamp_check(T1_received, label=""):
    """IAC verifies timestamp freshness"""
    T_now = get_timestamp()
    diff = T_now - T1_received
    fresh = diff < DELTA_T
    status = "✅ ACCEPTED (fresh)" if fresh else "❌ REJECTED (stale - REPLAY BLOCKED)"
    print(f"  {label} T_now - T1 = {diff}ms  → {status}")
    return fresh

# Test 1: Immediate replay (within window) — message passes timestamp check
print(f"\nScenario 1: Attacker replays IMMEDIATELY")
iac_timestamp_check(T1, "Immediate replay:")
print(f"  But attacker still cannot compute Ai = h(Ti ⊕ Ii ⊕ CK')")
print(f"  Because Ii and CK' are unknown → REPLAY STILL FAILS at Step 4")

# Test 2: Delayed replay (outside window)
print(f"\nScenario 2: Attacker waits 6 seconds then replays")
time_module.sleep(0.1)  # Simulate delay (we use 100ms for demo, paper uses ~5s threshold)
old_T1 = T1 - 10000  # Simulate a 10-second-old message
iac_timestamp_check(old_T1, "Delayed replay: ")

print(f"\n✅ PROPOSITION 2 PROVEN: Replay attack FAILS because:")
print(f"   1. Timestamps expire within ∆T = {DELTA_T}ms")
print(f"   2. Even within ∆T, attacker can't compute Ai without Ii")
print(f"   3. Each session uses fresh N1, N2 — old messages are useless")

---
# 🛡️ PROPOSITION 3: EBIAS Resists Man-in-the-Middle (MitM) Attacks

## What is a MitM Attack?
Attacker sits between Device and IAC, intercepts and possibly modifies all messages.

## How EBIAS Blocks It

> *"The computation of P1 and P3 requires the generation of random numbers N1 and N2, which A cannot replicate through eavesdropping and computation."*

Even if attacker collects `CK, P1, P2, P3, P4`:
- **P1 = N1 * G** → Cannot find N1 from P1 (ECDLP)
- **P3 = N2 * G** → Cannot find N2 from P3 (ECDLP)
- Cannot forge P4 = N2 * A'i without knowing N2
- Cannot compute the session key SK without N1 or N2

**The ECDLP (Elliptic Curve Discrete Logarithm Problem):**  
Given P1 = N1 * G, find N1. This is computationally infeasible for 160-bit keys. The best known algorithms take ~2^80 operations.

In [ ]:
# ============================================================
# CELL 7: Proposition 3 — MitM Attack Resistance
# ============================================================

print("=" * 60)
print("PROPOSITION 3: RESISTANCE TO MAN-IN-THE-MIDDLE ATTACKS")
print("=" * 60)

print("\n🔴 Attacker captures all public values:")
print(f"  P1 = N1 * G = {P1}")
print(f"  P2 = h(N1 * CK') = {hex(P2)[:20]}...")
print(f"  P3 = N2 * G = {P3}")
print(f"  P4 = N2 * A'i = {P4}")

print("\n--- Attacker tries to compute N1 from P1 (ECDLP) ---")
print(f"  P1.value = {hex(P1.value)[:20]}...")
print(f"  G.value  = {hex(G.value)[:20]}...")
print(f"  Need to find N1 such that N1 * G = P1")
print(f"  This is the ELLIPTIC CURVE DISCRETE LOG PROBLEM (ECDLP)")
print(f"  For 160-bit keys: brute force would take ~2^80 operations")
print(f"  At 10^12 guesses/second → would take {2**80 / 10**12 / (365*24*3600):.2e} YEARS")

print("\n--- Attacker tries to forge a valid session key ---")
print("  SK = h((N1 * P3) ∥ Ai ∥ T4)")
print("  Attacker needs N1 → IMPOSSIBLE (ECDLP)")
print("  Attacker needs Ai = h(Ti ⊕ Ii ⊕ CK') → IMPOSSIBLE (needs Ii and CK')")

print("\n✅ PROPOSITION 3 PROVEN: MitM attack FAILS because:")
print("   ECDLP makes it computationally infeasible to recover N1, N2 from P1, P3")

---
# 🛡️ PROPOSITION 4: EBIAS Resists Insider Attacks

## What is an Insider Attack?
A malicious employee or insider at the IAC tries to impersonate a legitimate device.

## How EBIAS Blocks It

> *"Within EBIAS, during the device registration phase, Di sends Ii to the IAC, where Ii = h(IDi∥PWi). Therefore, insiders cannot retrieve PWi through Ii, as the hash function is irreversible."*

The IAC never receives the raw password PWi — only `Ii = h(IDi ∥ PWi)`. Since SHA-256 is irreversible, even someone inside IAC cannot find PWi from Ii.

In [ ]:
# ============================================================
# CELL 8: Propositions 4-10 — Concise Proofs
# ============================================================

print("=" * 60)
print("PROPOSITIONS 4-10: VERIFICATION")
print("=" * 60)

# --- PROPOSITION 4: Insider Attack ---
print("\n[P4] INSIDER ATTACK RESISTANCE")
print(f"  Ii stored at IAC = h(IDi ∥ PWi) = {hex(Ii)[:20]}...")
print(f"  Insider tries: SHA256^-1({hex(Ii)[:20]}...) = ???")
print(f"  SHA-256 is IRREVERSIBLE → PWi CANNOT be recovered")
print(f"  ✅ P4 PROVEN")

# --- PROPOSITION 5: Session Key Theft ---
print("\n[P5] SESSION KEY THEFT RESISTANCE")
print(f"  Even if attacker gets SK = {hex(SK)[:20]}...")
print(f"  To impersonate device, need CK' = {CK_prime}")
print(f"  CK' is derived from CK = h(Ri ∥ X ∥ EXP_time ∥ PIDi)")
print(f"  Attacker needs X (IAC master secret) → IMPOSSIBLE")
print(f"  ✅ P5 PROVEN")

# --- PROPOSITION 6: Device Anonymity ---
print("\n[P6] DEVICE ANONYMITY")
print(f"  Transmitted PIDi = h(Ri ∥ IDIAC ∥ Ii) ⊕ IDIAC = {hex(PIDi)[:20]}...")
print(f"  Real IDi = '{IDi}' is NEVER transmitted")
print(f"  Ri is random → PIDi changes each registration → unlinkable!")
print(f"  ✅ P6 PROVEN")

# --- PROPOSITION 7: Forward Secrecy ---
print("\n[P7] PERFECT FORWARD SECRECY")
print(f"  SK = h((N1 * P3) ∥ Ai ∥ T4)")
print(f"  SK depends on N1, N2 (fresh randoms per session)")
print(f"  Even if X and PWi leaked later → past N1, N2 unknown → past SKs safe")
print(f"  ✅ P7 PROVEN")

# --- PROPOSITION 8: Mutual Authentication ---
print("\n[P8] MUTUAL AUTHENTICATION")
print(f"  Device proves identity to IAC: Y = h(P1∥P2∥T1), IAC verifies Y'==Y")
print(f"  Y'==Y: {Y == Y_prime}")
print(f"  IAC proves identity to Device: Z = h(P3∥P4∥T3), Device verifies Z'==Z")
print(f"  Z'==Z: {Z == Z_prime}")
print(f"  ✅ P8 PROVEN — BOTH parties authenticated each other")

# --- PROPOSITION 9: Cookie Theft ---
print("\n[P9] COOKIE THEFT RESISTANCE")
print(f"  CK = h(Rs ∥ X ∥ EXP_time ∥ PIDi) = {hex(CK)[:20]}...")
print(f"  CK is NEVER transmitted in plaintext over any channel")
print(f"  Device stores CK' = CK*G, not CK itself")
print(f"  Even with CK', attacker cannot reverse to get CK (ECDLP)")
print(f"  ✅ P9 PROVEN")

# --- PROPOSITION 10: KCI Attack ---
print("\n[P10] KEY COMPROMISE IMPERSONATION (KCI) RESISTANCE")
print(f"  Even if IAC private key X is compromised:")
print(f"  Attacker needs Ti = {hex(Ti)[:20]}... (stored on blockchain)")
print(f"  But Ti is only accessible via internal network")
print(f"  Cannot compute CK without Ti → Cannot impersonate device")
print(f"  ✅ P10 PROVEN")

print("\n" + "=" * 60)
print("ALL 10 PROPOSITIONS VERIFIED ✅")
print("=" * 60)

---
# 📊 Summary: All 10 Propositions at a Glance

| # | Proposition | Attack Type | Defense Mechanism |
|---|------------|-------------|--------------------|
| P1 | Password/Identity Guessing | Brute force on Ii | SHA-256 irreversibility + ECDLP |
| P2 | Replay Attack | Reuse old messages | Timestamps + ∆T check |
| P3 | Man-in-the-Middle | Intercept & modify | ECDLP hardness (N1, N2 unrecoverable) |
| P4 | Insider Attack | Insider uses Ii to get PWi | SHA-256 one-way function |
| P5 | Session Key Theft | Use stolen SK to impersonate | SK depends on CK' which needs X |
| P6 | Device Anonymity | Link traffic to real device | PIDi pseudonym changes each session |
| P7 | Forward Secrecy | Compromise past sessions | Per-session random N1, N2 |
| P8 | Mutual Authentication | Fake IAC or fake Device | Both sides verify hash chains |
| P9 | Cookie Theft | Steal CK from channel | CK never transmitted, only CK'=CK*G |
| P10 | KCI Attack | Compromise IAC private key X | Ti only on internal network |

---
## 🔑 Core Security Pillars

1. **SHA-256 is irreversible** → Protects passwords (P1, P4)
2. **ECDLP hardness** → Protects session randoms (P3, P5, P9)
3. **Timestamps** → Prevent replays (P2)
4. **Pseudonyms** → Protect identity (P6)
5. **Per-session randoms** → Forward secrecy (P7)
6. **Mutual verification** → Mutual auth (P8)
7. **Internal-only storage** → KCI protection (P10)